## Imports

In [1]:
import numpy as np
import pandas as pd
import json

import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import DataCollatorWithPadding

import evaluate

from transformers import TrainingArguments
from transformers import Trainer


## Setup

In [2]:
MODEL_NAME = "google-bert/bert-base-uncased"
SEED = 456                                  #42, 123, 456
NUM_LABELS = 3

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.14.0+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Load MNLI Dataset

In [3]:
mnli = load_dataset("nyu-mll/glue", "mnli")

train_dataset = mnli["train"]
val_matched = mnli["validation_matched"]
val_mismatched = mnli["validation_mismatched"]

print("Train:", len(train_dataset))
print("Validation matched:", len(val_matched))
print("Validation mismatched:", len(val_mismatched))

Train: 392702
Validation matched: 9815
Validation mismatched: 9832


In [4]:
mnli

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

## BERT Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Sample

In [6]:
example = train_dataset[0]

print("Premise:")
print(example["premise"])

print("\nHypothesis:")
print(example["hypothesis"])

Premise:
Conceptually cream skimming has two basic dimensions - product and geography.

Hypothesis:
Product and geography are what make cream skimming work. 


In [7]:
encoded = tokenizer(
    example["premise"],
    example["hypothesis"],
    truncation=True
)

encoded

{'input_ids': [101, 17158, 2135, 6949, 8301, 25057, 2038, 2048, 3937, 9646, 1011, 4031, 1998, 10505, 1012, 102, 4031, 1998, 10505, 2024, 2054, 2191, 6949, 8301, 25057, 2147, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
tokens

['[CLS]',
 'conceptual',
 '##ly',
 'cream',
 'ski',
 '##mming',
 'has',
 'two',
 'basic',
 'dimensions',
 '-',
 'product',
 'and',
 'geography',
 '.',
 '[SEP]',
 'product',
 'and',
 'geography',
 'are',
 'what',
 'make',
 'cream',
 'ski',
 '##mming',
 'work',
 '.',
 '[SEP]']

## Tokenization Function

In [9]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

## Tokenize Dataset

In [10]:
tokenized_mnli = mnli.map(
    tokenize_function,
    batched=True
)

In [11]:
tokenized_mnli

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9847
    })
})

In [12]:
tokenized_mnli["train"][0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1,
 'idx': 0,
 'input_ids': [101,
  17158,
  2135,
  6949,
  8301,
  25057,
  2038,
  2048,
  3937,
  9646,
  1011,
  4031,
  1998,
  10505,
  1012,
  102,
  4031,
  1998,
  10505,
  2024,
  2054,
  2191,
  6949,
  8301,
  25057,
  2147,
  1012,
  102],
 'token_type_ids': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1]}

In [13]:
print("Premise:")
print(tokenized_mnli["train"][0]["premise"])

print("\nHypothesis:")
print(tokenized_mnli["train"][0]["hypothesis"])

print("\nLabel:")
print(tokenized_mnli["train"][0]["label"])

print("\nTokens:")
print(
    tokenizer.convert_ids_to_tokens(
        tokenized_mnli["train"][0]["input_ids"]
    )
)

Premise:
Conceptually cream skimming has two basic dimensions - product and geography.

Hypothesis:
Product and geography are what make cream skimming work. 

Label:
1

Tokens:
['[CLS]', 'conceptual', '##ly', 'cream', 'ski', '##mming', 'has', 'two', 'basic', 'dimensions', '-', 'product', 'and', 'geography', '.', '[SEP]', 'product', 'and', 'geography', 'are', 'what', 'make', 'cream', 'ski', '##mming', 'work', '.', '[SEP]']


### Checking length

In [14]:
lengths = [
    len(
        tokenizer(
            premise,
            hypothesis,
            truncation=False
        )["input_ids"]
    )
    for premise, hypothesis in zip(
        mnli["train"]["premise"],
        mnli["train"]["hypothesis"]
    )
]

num_over_128 = sum(length > 128 for length in lengths)

print(f"Examples > 128 tokens: {num_over_128:,}")
print(f"Percentage truncated: {num_over_128 / len(lengths) * 100:.2f}%")
print(f"Longest example: {max(lengths)} tokens")

Examples > 128 tokens: 1,169
Percentage truncated: 0.30%
Longest example: 444 tokens


## Build FP32 BERT Classifier

In [15]:
label_names = mnli["train"].features["label"].names

id2label = {
    i: label
    for i, label in enumerate(label_names)
}

label2id = {
    label: i
    for i, label in enumerate(label_names)
}

print(id2label)
print(label2id)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}
{'entailment': 0, 'neutral': 1, 'contradiction': 2}


In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
print(model.config)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "entailment",
    "1": "neutral",
    "2": "contradiction"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "contradiction": 2,
    "entailment": 0,
    "neutral": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.17.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [18]:
print(model.classifier)

Linear(in_features=768, out_features=3, bias=True)


## Dynamic Padding

In [19]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

## Defining Evaluation

In [20]:
accuracy_metric = evaluate.load("accuracy")

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )

## Hyperparameters

In [22]:
LEARNING_RATE = 2e-5
BATCH_SIZE = 32
NUM_EPOCHS = 3
#WEIGHT_DECAY = 0.01

In [23]:
training_args = TrainingArguments(
    output_dir="../models/fp32_seed456",

    learning_rate=LEARNING_RATE,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    num_train_epochs=NUM_EPOCHS,
    #weight_decay=WEIGHT_DECAY,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=500,

    seed=SEED
)

## Trainer

In [24]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_mnli["train"],
    eval_dataset=tokenized_mnli["validation_matched"],

    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [25]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [26]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.459394,0.413481,0.841875
2,0.338605,0.421892,0.847886
3,0.235237,0.492026,0.844218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Analysis

In [27]:
peak_memory = torch.cuda.max_memory_allocated() / 1024**3

print(f"Peak GPU memory: {peak_memory:.2f} GB")

Peak GPU memory: 3.77 GB


In [28]:
train_result

TrainOutput(global_step=36816, training_loss=0.36518239933529506, metrics={'train_runtime': 5106.03, 'train_samples_per_second': 230.728, 'train_steps_per_second': 7.21, 'total_flos': 5.433690035127514e+16, 'train_loss': 0.36518239933529506, 'epoch': 3.0})

In [29]:
trainer.state.log_history

[{'loss': 0.8240504150390625,
  'grad_norm': 7.203640937805176,
  'learning_rate': 1.972892220773577e-05,
  'epoch': 0.04074315514993481,
  'step': 500},
 {'loss': 0.6652103271484375,
  'grad_norm': 7.2250823974609375,
  'learning_rate': 1.945730117340287e-05,
  'epoch': 0.08148631029986962,
  'step': 1000},
 {'loss': 0.5990796508789062,
  'grad_norm': 7.380050182342529,
  'learning_rate': 1.9185680139069973e-05,
  'epoch': 0.12222946544980444,
  'step': 1500},
 {'loss': 0.5800201416015625,
  'grad_norm': 6.883297920227051,
  'learning_rate': 1.891405910473707e-05,
  'epoch': 0.16297262059973924,
  'step': 2000},
 {'loss': 0.55492236328125,
  'grad_norm': 8.223471641540527,
  'learning_rate': 1.8642438070404173e-05,
  'epoch': 0.20371577574967406,
  'step': 2500},
 {'loss': 0.5435216064453126,
  'grad_norm': 5.45443058013916,
  'learning_rate': 1.8370817036071274e-05,
  'epoch': 0.24445893089960888,
  'step': 3000},
 {'loss': 0.528676513671875,
  'grad_norm': 8.585277557373047,
  'lear

In [30]:
matched_results = trainer.evaluate(
    eval_dataset=tokenized_mnli["validation_matched"],
    metric_key_prefix="mnli_matched"
)

matched_results

Training Loss,Validation Loss,Epoch,Matched Loss,Matched Accuracy
0.235237,No log,3,0.492026,0.844218


{'mnli_matched_loss': 0.4920262098312378,
 'mnli_matched_accuracy': 0.8442180336220071}

In [31]:
mismatched_results = trainer.evaluate(
    eval_dataset=tokenized_mnli["validation_mismatched"],
    metric_key_prefix="mnli_mismatched"
)

mismatched_results

Training Loss,Validation Loss,Epoch,Mismatched Loss,Mismatched Accuracy
0.235237,No log,3,0.475314,0.847335


{'mnli_mismatched_loss': 0.4753141701221466,
 'mnli_mismatched_accuracy': 0.8473352318958502}

In [32]:
FINAL_MODEL_DIR = "../models/fp32_seed456/final"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print(f"FP32 model saved to: {FINAL_MODEL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FP32 model saved to: ../models/fp32_seed456/final


In [33]:
log_df = pd.DataFrame(trainer.state.log_history)

TRAINING_LOG_PATH = "../results/mnli/fp32_seed456_training_log.csv"
log_df.to_csv(TRAINING_LOG_PATH, index=False)

print(f"Training log saved to: {TRAINING_LOG_PATH}")

log_df.tail()

Training log saved to: ../results/mnli/fp32_seed456_training_log.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_accuracy,eval_runtime,eval_samples_per_second,eval_steps_per_second,...,mnli_matched_loss,mnli_matched_accuracy,mnli_matched_runtime,mnli_matched_samples_per_second,mnli_matched_steps_per_second,mnli_mismatched_loss,mnli_mismatched_accuracy,mnli_mismatched_runtime,mnli_mismatched_samples_per_second,mnli_mismatched_steps_per_second
74,0.235237,6.687324,1.722077e-07,2.97425,36500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,NaN,NaN,NaN,3.00000,36816,0.492026,0.844218,12.524,783.692,24.513,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,NaN,NaN,NaN,3.00000,36816,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,NaN,NaN,NaN,3.00000,36816,NaN,NaN,NaN,NaN,NaN,...,0.492026,0.844218,12.4853,786.124,24.589,NaN,NaN,NaN,NaN,NaN
78,NaN,NaN,NaN,3.00000,36816,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.475314,0.847335,12.9755,757.734,23.737


In [34]:
mnli_results = {
    "matched": matched_results,
    "mismatched": mismatched_results,
}

RESULTS_PATH = "../results/mnli/fp32_seed456_results.json"

with open(RESULTS_PATH, "w") as f:
    json.dump(mnli_results, f, indent=4)

print(f"MNLI results saved to: {RESULTS_PATH}")

MNLI results saved to: ../results/mnli/fp32_seed456_results.json


In [35]:
trainer.save_state()

In [36]:
run_info = {
    "model": MODEL_NAME,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "peak_gpu_memory_gb": peak_memory,
}

RUN_INFO_PATH = "../results/mnli/fp32_seed456_run_info.json"

with open(RUN_INFO_PATH, "w") as f:
    json.dump(run_info, f, indent=4)

run_info

{'model': 'google-bert/bert-base-uncased',
 'seed': 456,
 'max_length': 128,
 'learning_rate': 2e-05,
 'batch_size': 32,
 'num_epochs': 3,
 'peak_gpu_memory_gb': 3.767009735107422}